# Animation Engine — selected humanoid → ARDY motion → validated FBX

This engine is independent from the 3D Engine. Upload only the humanoid you actually want to animate.

Pipeline: `character → Make-It-Animatable rig → ARDY Core motion → validated FBX bridge → Auto-Rig-Pro retarget → contract validation → Unreal package`

If the input is a TRELLIS GLB, the notebook also preserves a copy as the **PBR/material master** because FBX should be treated primarily as the skeletal/animation carrier.


In [ ]:
import shutil, subprocess, pathlib

r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    text=True, capture_output=True, check=True
).stdout.strip()
name, mem = [x.strip() for x in r.splitlines()[0].rsplit(",", 1)]
free = shutil.disk_usage("/content").free / 1024**3
print(f"GPU: {name} | VRAM: {int(mem)/1024:.1f} GiB | Free disk: {free:.1f} GiB")
if int(mem) < 24000:
    raise RuntimeError("Use a >=24 GB GPU for the supported combined Animation Engine.")
if free < 30:
    raise RuntimeError("Need >=30 GiB free disk.")


In [ ]:
import pathlib, shutil
if pathlib.Path("/content/My-works").exists():
    shutil.rmtree("/content/My-works")
!git clone -q --depth 1 https://github.com/Logan17de/My-works.git /content/My-works

TOOLS = "/content/My-works/ai-3d-animation-engines/animation-engine"
OUTPUT_DIR = "/content/animation_outputs"
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
!bash {TOOLS}/install_animation.sh


In [ ]:
import getpass, os, pathlib, shutil
from google.colab import files

token = getpass.getpass("Hugging Face token (Llama 3 access): " ).strip()
if not token:
    raise ValueError("HF token required")
os.environ["HF_TOKEN"] = token

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one humanoid GLB/FBX/OBJ/PLY.")
target_name = next(iter(uploaded))
TARGET_CHARACTER = f"/content/{target_name}"
suffix = pathlib.Path(TARGET_CHARACTER).suffix.lower()
if suffix not in {".glb", ".fbx", ".obj", ".ply"}:
    raise ValueError("Unsupported character format")

MATERIAL_SOURCE = None
if suffix == ".glb":
    MATERIAL_SOURCE = f"{OUTPUT_DIR}/character_material_source.glb"
    shutil.copy2(TARGET_CHARACTER, MATERIAL_SOURCE)
    print("PBR material master:", MATERIAL_SOURCE)

PROMPT = "A person walks forward, stops, and waves with the right hand." #@param {type:"string"}
DURATION_SECONDS = 6.0 #@param {type:"number"}
SEED = 0 #@param {type:"integer"}
TARGET_ALREADY_RIGGED = False #@param {type:"boolean"}
MIA_NO_FINGERS = True #@param {type:"boolean"}


In [ ]:
import subprocess, shlex, os, pathlib, shutil, numpy as np, json

motion_stem = f"{OUTPUT_DIR}/motion"
cmd = [
    "/opt/conda/bin/conda", "run", "-n", "ardy", "python", "scripts/generate.py",
    PROMPT, "--model", "core", "--duration", str(DURATION_SECONDS),
    "--seed", str(SEED), "--output", motion_stem,
]
print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd="/content/ardy", env=os.environ.copy(), check=True)

MOTION_NPZ = f"{motion_stem}.npz"
MOTION_BRIDGE = f"{OUTPUT_DIR}/motion_bridge.npz"
MOTION_PREVIEW = f"{OUTPUT_DIR}/motion_preview.mp4"

subprocess.run([
    "/opt/conda/bin/conda", "run", "-n", "ardy", "python",
    f"{TOOLS}/enrich_ardy_motion.py", "--input", MOTION_NPZ, "--output", MOTION_BRIDGE
], check=True)

with np.load(MOTION_BRIDGE, allow_pickle=True) as z:
    MOTION_FPS = float(np.asarray(z["fps"]).reshape(-1)[0])
if MOTION_FPS <= 0:
    raise RuntimeError(f"Invalid ARDY FPS: {MOTION_FPS}")
print("ARDY FPS contract:", MOTION_FPS)

subprocess.run([
    "/opt/conda/bin/conda", "run", "-n", "ardy", "python",
    f"{TOOLS}/preview_ardy_motion.py", "--input", MOTION_BRIDGE, "--output", MOTION_PREVIEW
], check=True)

from IPython.display import Video, display
display(Video(MOTION_PREVIEW, embed=True))

ARDY_SOURCE_FBX = f"{OUTPUT_DIR}/ardy_source.fbx"
subprocess.run([
    "/opt/conda/bin/conda", "run", "-n", "mia", "python",
    f"{TOOLS}/ardy_motion_to_fbx.py", "--input", MOTION_BRIDGE, "--output", ARDY_SOURCE_FBX
], check=True)

RIGGED_TARGET = f"{OUTPUT_DIR}/character_rigged.fbx"
if TARGET_ALREADY_RIGGED:
    if pathlib.Path(TARGET_CHARACTER).suffix.lower() != ".fbx":
        raise ValueError("Skip-rig requires FBX")
    shutil.copy2(TARGET_CHARACTER, RIGGED_TARGET)
else:
    rig_cmd = [
        "/opt/conda/bin/conda", "run", "-n", "mia", "python",
        f"{TOOLS}/rig_character_mia.py", "--input", TARGET_CHARACTER, "--output", RIGGED_TARGET
    ]
    if MIA_NO_FINGERS:
        rig_cmd.append("--no-fingers")
    subprocess.run(rig_cmd, env={**os.environ, "MIA_ROOT": "/content/Make-It-Animatable"}, check=True)

FINAL_FBX = f"{OUTPUT_DIR}/character_animated.fbx"
FINAL_GLB = f"{OUTPUT_DIR}/character_animated_preview.glb"
subprocess.run([
    "/opt/conda/bin/conda", "run", "-n", "mia", "python",
    f"{TOOLS}/retarget_with_mia.py",
    "--target", RIGGED_TARGET,
    "--animation", ARDY_SOURCE_FBX,
    "--output", FINAL_FBX,
    "--preview-glb", FINAL_GLB,
    "--fps", str(MOTION_FPS),
], env={**os.environ, "MIA_ROOT": "/content/Make-It-Animatable"}, check=True)

CONTRACT_REPORT = f"{OUTPUT_DIR}/animation_contract_report.json"
subprocess.run([
    "/opt/conda/bin/conda", "run", "-n", "mia", "python",
    f"{TOOLS}/validate_animation_contract.py",
    "--character-source", TARGET_CHARACTER,
    "--rigged-target", RIGGED_TARGET,
    "--source-animation", ARDY_SOURCE_FBX,
    "--animated-target", FINAL_FBX,
    "--expected-fps", str(MOTION_FPS),
    "--report", CONTRACT_REPORT,
    "--strict",
], check=True)

report = json.loads(pathlib.Path(CONTRACT_REPORT).read_text())
print("Contract passed:", report["passed"])
if report["warnings"]:
    print("Warnings:", *report["warnings"], sep="\n- " )


In [ ]:
import pathlib, shutil, json
from google.colab import files

package_dir = pathlib.Path(f"{OUTPUT_DIR}/unreal_package")
package_dir.mkdir(exist_ok=True)

for path in [FINAL_FBX, CONTRACT_REPORT, MOTION_PREVIEW]:
    shutil.copy2(path, package_dir / pathlib.Path(path).name)
if MATERIAL_SOURCE:
    shutil.copy2(MATERIAL_SOURCE, package_dir / pathlib.Path(MATERIAL_SOURCE).name)

package_manifest = {
    "animation_master": "character_animated.fbx",
    "material_master": "character_material_source.glb" if MATERIAL_SOURCE else None,
    "motion_preview": "motion_preview.mp4",
    "contract_report": "animation_contract_report.json",
    "fps": MOTION_FPS,
    "unreal_note": "Import FBX as Skeletal Mesh/Animation. Use automatic/custom sample rate matching fps. Reapply PBR from material master GLB when present."
}
(package_dir / "package_manifest.json").write_text(json.dumps(package_manifest, indent=2))

archive = shutil.make_archive(f"{OUTPUT_DIR}/unreal_character_package", "zip", package_dir)
print("Package:", archive)
files.download(archive)


### Final contract

The ZIP is the handoff:
- `character_animated.fbx` — skeletal/animation master
- `character_material_source.glb` — original PBR master when the input was GLB
- `animation_contract_report.json` — scale/FPS/motion-transfer validation
- `package_manifest.json`
- `motion_preview.mp4`

The validator deliberately compares **motion deltas**, not raw character joint positions, so different body proportions do not automatically count as retarget errors.
